## Day 2: EDA

Findings so far:
- `movies.id` has 3 corrupted rows (columns shifted). Drop before merge
- `movies.overview` has 954 nulls: need fallback for content-based filtering
- `links_kaggle.tmdbId` has 219 nulls: ~2% of MovieLens movies won't join to Kaggle metadata
- `keywords`, `credits`, `genres` columns are stringified lists. Need parsing later

In [1]:
import os
print(os.getcwd())

/Users/lea/Documents/Projects/movie-recommender/notebooks


In [2]:
import os

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

print("Working directory is now:", os.getcwd())

Working directory is now: /Users/lea/Documents/Projects/movie-recommender


In [3]:
import pandas as pd

# MovieLens
ratings = pd.read_csv('data/ratings.csv')
links_ml = pd.read_csv('data/links_movielens.csv')

# Kaggle
movies = pd.read_csv('data/movies_metadata.csv', low_memory=False)
keywords = pd.read_csv('data/keywords.csv')
credits = pd.read_csv('data/credits.csv')
links_kaggle = pd.read_csv('data/links_kaggle.csv')

In [4]:
for name, df in [('ratings', ratings), ('links_ml', links_ml), 
                  ('movies', movies), ('keywords', keywords), 
                  ('credits', credits), ('links_kaggle', links_kaggle)]:
    print(f"{name}: {df.shape}")

ratings: (100836, 4)
links_ml: (9742, 3)
movies: (45466, 24)
keywords: (46419, 2)
credits: (45476, 3)
links_kaggle: (45843, 3)


In [5]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


In [6]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45466 non-null  object 
 1   belongs_to_collection  4494 non-null   object 
 2   budget                 45466 non-null  object 
 3   genres                 45466 non-null  object 
 4   homepage               7782 non-null   object 
 5   id                     45466 non-null  object 
 6   imdb_id                45449 non-null  object 
 7   original_language      45455 non-null  object 
 8   original_title         45466 non-null  object 
 9   overview               44512 non-null  object 
 10  popularity             45461 non-null  object 
 11  poster_path            45080 non-null  object 
 12  production_companies   45463 non-null  object 
 13  production_countries   45463 non-null  object 
 14  release_date           45379 non-null  object 
 15  re

In [ ]:
movies[pd.to_numeric(movies['id'], errors='coerce').isna()] #converts id to number + checks for NaN values 

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
19730,- Written by Ørnås,0.065736,/ff9qCepilowshEtG2GYWwzt2bs4.jpg,"[{'name': 'Carousel Productions', 'id': 11176}...","[{'iso_3166_1': 'CA', 'name': 'Canada'}, {'iso...",1997-08-20,0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,...,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29503,Rune Balot goes to a casino connected to the ...,1.931659,/zV8bHuSL6WXoD6FWogP9j4x80bL.jpg,"[{'name': 'Aniplex', 'id': 2883}, {'name': 'Go...","[{'iso_3166_1': 'US', 'name': 'United States o...",2012-09-29,0,68.0,"[{'iso_639_1': 'ja', 'name': '日本語'}]",Released,...,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35587,Avalanche Sharks tells the story of a bikini ...,2.185485,/zaSf5OG7V8X8gqFvly88zDdRm46.jpg,"[{'name': 'Odyssey Media', 'id': 17161}, {'nam...","[{'iso_3166_1': 'CA', 'name': 'Canada'}]",2014-01-01,0,82.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,...,22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
keywords.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46419 entries, 0 to 46418
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        46419 non-null  int64 
 1   keywords  46419 non-null  object
dtypes: int64(1), object(1)
memory usage: 725.4+ KB


In [9]:
credits.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45476 entries, 0 to 45475
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   cast    45476 non-null  object
 1   crew    45476 non-null  object
 2   id      45476 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 1.0+ MB


In [10]:
links_kaggle.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45843 entries, 0 to 45842
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   movieId  45843 non-null  int64  
 1   imdbId   45843 non-null  int64  
 2   tmdbId   45624 non-null  float64
dtypes: float64(1), int64(2)
memory usage: 1.0 MB


In [ ]:
import pandas as pd

# MovieLens
ratings = pd.read_csv('data/ratings.csv')
links_ml = pd.read_csv('data/links_movielens.csv')

# Kaggle
movies = pd.read_csv('data/movies_metadata.csv', low_memory=False)
keywords = pd.read_csv('data/keywords.csv')
credits = pd.read_csv('data/credits.csv')
links_kaggle = pd.read_csv('data/links_kaggle.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'data/ratings.csv'